# 03 — Embeddings e indexação vetorial

## Objetivo
Converter os documentos preparados em embeddings multilíngues, construir o índice FAISS e validar sua persistência e recuperação.

A implementação reutilizada está em:
- `src/embeddings/embedder.py`;
- `src/retrieval/retriever.py`;
- `src/data/loader.py` e `src/data/preprocessing.py`.

A geração de embeddings e a criação do índice podem levar alguns minutos e exigem memória disponível. O modelo é baixado na primeira execução, caso ainda não esteja no cache local.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
from src.data.loader import load_olist_raw
from src.data.preprocessing import enrich_reviews, build_documents
from src.embeddings.embedder import EmbeddingModel
from src.retrieval.retriever import VectorRetriever
from config.settings import settings

INDEX_DIR = settings.data_dir / "vectorstore"
raw = load_olist_raw(RAW_DIR)
prepared = enrich_reviews(raw)
documents = build_documents(prepared)

print(f"Documentos para indexação: {len(documents):,}")


Documentos para indexação: 40,950


## 1. Modelo de embeddings

O modelo padrão é `paraphrase-multilingual-MiniLM-L12-v2`, selecionado por oferecer embeddings multilíngues com execução local.

In [2]:
embedder = EmbeddingModel(settings.embedding_model)
print("Modelo:", settings.embedding_model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 2. Geração dos vetores

Cada comentário é convertido em um vetor. A implementação normaliza os embeddings para permitir busca por produto interno equivalente à similaridade cosseno.

In [3]:
texts = [doc["text"] for doc in documents]
embeddings = embedder.encode(texts, batch_size=64)

print("Formato dos embeddings:", embeddings.shape)
print("Quantidade de documentos:", len(documents))
assert embeddings.shape[0] == len(documents)


Batches:   0%|          | 0/640 [00:00<?, ?it/s]

Formato dos embeddings: (40950, 384)
Quantidade de documentos: 40950


## 3. Construção do índice FAISS

O índice `IndexFlatIP` realiza busca exata por produto interno. Para os vetores normalizados, o score corresponde à similaridade cosseno.

In [4]:
retriever = VectorRetriever(embeddings, documents)
print("Tipo do índice:", type(retriever.index).__name__)
print("Vetores indexados:", retriever.size)


Tipo do índice: IndexFlatIP
Vetores indexados: 40950


## 4. Persistência e recarga

São persistidos o índice FAISS e os documentos/metadados correspondentes.

In [5]:
retriever.save(INDEX_DIR)
loaded_retriever = VectorRetriever.load(INDEX_DIR)

print("Diretório:", INDEX_DIR)
print("Vetores recarregados:", loaded_retriever.size)
print("Documentos recarregados:", len(loaded_retriever.documents))
assert loaded_retriever.size == len(documents)
assert len(loaded_retriever.documents) == len(documents)


Diretório: F:\FIAP\TechChallenge-Fase4\data\vectorstore
Vetores recarregados: 40950
Documentos recarregados: 40950


## 5. Teste de recuperação semântica

Faça uma consulta simples e inspecione os textos, identificadores e scores retornados pelo índice.

In [6]:
question = "Quais são os principais problemas relatados pelos clientes sobre a entrega?"
query_vector = embedder.encode([question])[0]
results = loaded_retriever.search(
    query_vector,
    top_k=settings.top_k,
    min_score=settings.min_relevance_score,
)

print("Pergunta:", question)
print("Resultados:", len(results))
for item in results:
    print(f"\n[{item['document_id']}] score={item['score']:.4f}")
    print("Nota:", item.get("metadata", {}).get("review_score"))
    print(item["text"])


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Pergunta: Quais são os principais problemas relatados pelos clientes sobre a entrega?
Resultados: 5

[42a3a3e2b79465364e24f6f4c642bc23] score=0.7341
Nota: 1
Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto entregue, porém foi entregue apenas 1 produto, aguardo solução!

[e06d9a717c61331208eabed007431c41] score=0.7188
Nota: 4
Embora tenha havido problemas com a entrega pelos correios o produto chegou.

[949eca021ed60d00a2d2bda3cc686d4d] score=0.7139
Nota: 1
muitos desencontros nas entregas de produtos

[7d8d252cc0ae002b13db033ec1cd9dcb] score=0.6987
Nota: 1
Demora de entrega

[a553875413fd30a7f78cd701b0064a6f] score=0.6932
Nota: 5
Recomendo o produto e o fornecedor, pois o atraso na entrega foi causado pelos serviços dos Correios.


## 6. Conclusões da indexação

Registre o modelo, a dimensão dos vetores, o total indexado, o tempo aproximado de processamento e o resultado da consulta de inspeção.

**Próxima etapa:** [04 — Avaliação do RAG](04_avaliacao_rag.ipynb).